
# 1. 형태소 분석 기반 토큰화의 문제

- 형태소 분석 기반 토큰화는 **"이미 알고 있는 단어"** 를 중심으로 판단한다. 그래서 오탈자나 띄어쓰기 실수, 신조어, 외래어, 고유어에 대해서 취약한 단점이 있다.
- 이로 인해 발생 할 수 있는 잠재적 문제점은:
    - 어휘사전이 불필요하게 커진다.
        - 같은 의미의 단어가 형태소 분석이 안되어 여러개 등록될 수있다.
        - ex) 신조어 `돈쭐` 이라는 단어를 인식 못할 경우 `"돈쭐내러", "돈쭐나", "돈쭐냄"` 등이 다 등록 될 수 있다.
    - OOV(Out Of Vocab)에 대응하기 어렵게 만든다.
        - 같은 어근의 단어가 있지만 조사등이 바뀐 신조어등을 OOV로 인식할 수있다.


## 1.1 어휘 사전(Vocabulary)과 Out Of Vocabulary (OOV)

- 어휘사전(Vocab)은 토크나이저(Tokenizer)가 사용하는 모든 토큰의 집합이며,**각 토큰**을 고유한 **정수 ID**에 매핑한 사전이다. 토크나이저가 텍스트를 토큰 ID 시퀀스로 변환할 때 기준으로 사용된다. 

   - 매핑된 정수는 모델에 입력되는 텍스트 데이터를 숫자 형식으로 변환해 모델이 처리할 수 있도록 돕는다.
   - 예
        ```bash
        {
            "자연어": 0,
            "처리": 1,
            "는": 2,
            "재미있다": 3,
            "공부": 4,
        }
        ```
- **Out Of Vocabulary (OOV)**
   - 어휘 사전(Vocab): 코퍼스를 구성하는 모든 토큰의 집합.
   - **OOV**란 어휘 사전에 포함되지 않은 토큰을 의미하며, 모델이 해당 토큰을 처리할 수 없기 때문에 일반적으로 특별한 토큰(예: `[UNK]`)으로 대체되거나 다른 방식으로 처리된다.

# 2. Subword Tokenization(하위 단어 토큰화)

## 2.1 정의

- Subword Tokenization은 단어를 더 작은 단위(subword)로 나누어 텍스트를 토큰화하는 방식이다.  
    - subword는 하나의 단어를 구성하는 단어들을 말한다.(coworker: co, work, er)
- 주로 자주 등장하는 단어의 일부를 공통된 토큰으로 만들고, 희귀하거나 복합적인 단어는 작은 조각(subword)으로 나누어 처리한다.
- 단어 자체를 그대로 사용하기보다는 단어의 일부를 나누어 처리함으로써 새로운 단어나 미등록 단어(Out-of-Vocabulary) 문제를 줄일 수 있다.

## 2.2 장점

1. **미등록 단어 처리 가능**  
   -  새로운 단어(신조어, 속어, 고유어등)가 등장해도 미리 정의된 subword를 조합해서 표현할 수 있어 OOV 문제를 줄일 수 있다.  

2. **어휘 크기 축소**  
   - 같은 subword를 여러 단어에서 공유함으로써, 완전한 단어를 사용하는 경우보다 어휘집의 크기를 작게 유지할 수 있다.


## 2.3 종류

1. **Byte-Pair Encoding (BPE)**  
   - 자주 등장하는 문자 쌍을 반복적으로 병합해 서브워드를 생성하는 방식.
   - OpenAI의 GPT 모델에 사용된 토크나이저이다.

2. **Unigram**  
   - 빈도기반 확률모델에 따라 subword 단위를 선택하는 방식이다.  
   - BPE보다 유연하여 더 다양한 분할 결과를 얻을 수 있다.

3. **WordPiece**  
   - BPE와 유사하지만, 빈도수가 아니라, 가능성이 높은 조합(합쳐질 가능성이 높은 subword)에 기반해 subword들을 찾는다.
   - Google의 BERT 모델에 사용된 토크나이저이다.

# 3. Byte Pair Encoding 방식

- 원래 Text data 압축을 위해 만들어진 방법으로 text 에서 자주 같이 나오는 두 글짜 쌍을 합쳐서 하나의 부호(기호,글자)로 만들어 나가면서 글자 수를 줄이는 알고리즘이다. 
- 연속된 글자 쌍이 더 나타나지 않거나 정해진 어휘사전 크기에 도달 할 때 까지 조합을 찾아 부호화 하는 작업을 반복한다.

## 3.1 text 압축 방식의 예
- 원문: abracadabra
1. AracadAra: ab -> A :=> 원문에서 가장 빈도수 많은 ab를 A(부호로 아무 글자나 사용할 수 있다.)로 치환
2. ABcadAB: ra -> B :=> 1에서 가장 빈도수가 많은 ra를 B로 치환
3. CcadC: AB -> C :=> 2에서 가장 빈도수 맣은 AB를 C로 치환한다.(치환된 글자 쌍도 변환대상에 포함된다.)

## 3.2 BPE Tokenizer 방식
BPE 토크나이저는 자주 등장하는 글자 쌍을 찾아 치환하는 대신 **단어 사전**에 추가한다.

### 3.2. 예)
1. 1차적으로 토큰화를 진행한다. 일반적으로 공백 기준으로 토큰화를 한다.
    - BPE 방식은 1차적으로 진행한 토큰 내에서 글자쌍을 만들어 어휘사전에 등록한다.
    - 1차 토큰화 결과가 다음과 같다고 가정하자.
      - 각 토큰과 빈도수
        - ('low', 5), ('lower', 2), ('newest', 6), ('widest', 3) 
      - 'low', 'lower', 'newest', 'widest' 로 구성됨.
2. 모든 토큰들을 글자 단위로 나누고 어휘사전에 추가한다.
    - ('l', 'o', 'w',  5), ('l', 'o', 'w', 'e', 'r', 2), ('n', 'e', 'w', 'e', 's', 't', 6), ('w', 'i', 'd', 'e', 's', 't', 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
3. 토큰 별로 가장 자주 등장하는 글자 쌍(byte pair)를 찾는다.  위에서는 **'es'가 총 9번으로 가장 많이 등장함**. 'e'와 's'를 'es'로 합치고 어휘 사전에 추가한다.
    - ('l', 'o', 'w',  5), ('l', 'o', 'w', 'e', 'r', 2), ('n', 'e', 'w', **'es'**, 't', 6), ('w', 'i', 'd', **'es'**, 't', 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'es'**]
4. 3 번의 과정을 계속 반복한다. 빈도수가 가장 많은 'es'와 't' 쌍을 'est'로 병합하고 'est'를 어휘 사전에 추가한다.
    - ('l', 'o', 'w',  5), ('l', 'o', 'w', 'e', 'r', 2), ('n', 'e', 'w', **'est'**, 6), ('w', 'i', 'd', **'est'**, 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'es'**, **'est'**]
5. 만약 10번 반복했다고 하면 다음과 같은 빈도 사전과 어휘 사전이 생성된다.
    - 빈도 사전: (**'low'**, 5), (**'low'**, 'e', 'r', 2), ('n', 'e', 'w', **'est'**, 6), ('w', 'i', 'd', **'est'**, 3)
    - 어휘사전: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'es'**, **'est'**, **'lo'**,**'low'**, **'ne'**, **'new'**, **'newest'**, **'wi'**, **'wid'**, **'widest'**]

- 위와 같이 어휘 사전이 만들어 지면 원래 어휘서전에 없던 것들에 대한 처리를 할 수있다.
    - ex)
        - 'newer' :=> 'new', 'e', 'r', 
        - 'lowest' :=> 'low', 'est'
        - 'wider' :=> 'wid', 'e', 'r'

# 4. WordPiece tokenizer

- Byte Pair Encoding 이 빈도 기반이라면 wordpiece tokenizer는 확률 기반으로 글자 쌍을 병합한다.
- 두개 글자 쌍의 빈도수를 각 개별 글자 빈도수의 곱으로 나눈 점수가 가장 높은 순서대로 글자쌍을 묶어 나간다.

$$
score = \cfrac{f(x, y)}{f(x)\cdot f(y)} 
$$

함수 f는 빈도를 나타내며 x, y는 병합하려는 하위 단어이다.

- 각 토큰을 글자 단위로 분해 후 어휘사전에 추가
  -  ('l','o','w', 5), ('l','o','w', 'e', 'r', 2), ('n', 'e', 'w', 'e', 's', 't', 6), ('w', 'i', 'd', 'e', 's', 't', 3)
  - 어휘사전: ('d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w')
- 가장 빈도수가 높은 쌍은 'e','s'로 9번 등장한다. 이때 각 글자는 전체에서 각각 'e'는 17번, 's'는 9번 등장한다. 위 공식에 대입하면 score는 $\frac{9}{17 \times 9} \approx 0.06$ 이다.
- 'i'와 'd' 쌍은 3번만 등장하지만 전체에서 각각 'i' 3번, 'd' 3번 등장한다. 그래서 score는 $\frac{3}{3 \times 3} \approx 0.33$ 이다.
- 나타난 빈도수는 'es' 가 많치만 더 높은 score를 가지는 'id' 쌍을 병합한다.
  -  ('l','o','w', 5), ('l','o','w', 'e', 'r', 2), ('n', 'e', 'w', 'e', 's', 't', 6), ('w', **'id'**, 'e', 's', 't', 3)
  - 어휘사전: ('d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', **'id'**)
위의 작업을 반복해 연속된 글자 쌍이 더이상 나타나지 않거나 어휘 사전 max 크기에 도달할 때 까지 학습한다.

# 5. Unigram 방식
- 빈도 기반 확률 모델을 사용하여 효율적으로 서브워드를 선택하고, 불필요한 서브워드를 제거해 최적의 어휘 크기를 찾는 알고리즘

- **초기 어휘 집합 구성**
    - 대상 text에 모든 단어와 그 서브스트링을 포함한 어휘 집합을 생성한다. 이 어휘 집합은 나올 수있는 모든 subword들을 다 모아놓은 것이다. 
    - 예를 들어 "hug" 단어의  ["h", "u", "g", "hu", "ug", "hug"]  substring을 만든다. 이들이 subword 후보가 된다.
- **각 Subword의 빈도수 기반 확률 계산**
    -  $\cfrac{subword가\;나타난\;횟수}{전체\;빈도수}$ 로 각 subword들의 나타난 확률을 계산한다.
- **가능한 분할에 대한 확률 계산**
    - 단어를 여러 서브워드로 분할할 수 있는 경우, 각 분할에 대한 전체 확률을 계산한다.
    - 확률 계산은 $ P(subword1)\;\times \; P(subword2)\;\times\; ..$ 으로 계산한다.
    - 예를 들어 "hug" 를 분할 한다고 했을 때
        1. \["h", "u", "g"\]: $ P(h) \times P(u) \times P(g) $
        2. \["hu", "g"\]: $ P(hu) \times P(g) $

   - 각각의 확률을 계산한 후, **가장 높은 확률**을 가진 분할을 선택한다.
     - 위 예에서 만약 1의 확률이 0.01 이고 2의 확률이 0.00001 이라면 첫번째 분할이 선택된다.

- **서브워드 제거**
    - 위의 훈련과정에서 불필요한 서브워드를 제거하면서 최적의 어휘 집합을 찾아간다. 
    - 제거 대상은 빈도수가 낮거나 조합에 크게 영향을 주지 않은 subword들이다.

> ### korpora 말뭉치
> - 다양한 한글 데이터셋을 제공하는 패키지
> - `pip install korpora`

In [3]:
from Korpora import Korpora
# 청화대 국민 청원 데이터셋
corpus = Korpora.load("korean_petitions")
corpus


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : Hyunjoong Kim lovit@github
    Repository : https://github.com/lovit/petitions_archive
    References :

    청와대 국민청원 게시판의 데이터를 월별로 수집한 것입니다.
    청원은 게시판에 글을 올린 뒤, 한달 간 청원이 진행됩니다.
    수집되는 데이터는 청원종료가 된 이후의 데이터이며, 청원 내 댓글은 수집되지 않습니다.
    단 청원의 동의 개수는 수집됩니다.
    자세한 내용은 위의 repository를 참고하세요.

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[korean_petitions] download petitions_2017-08: 1.84MB [00:00, 7.40MB/s]                            
[korean_petitions] download petitions_2017-09: 20.4MB [00:00, 23.1MB/s]                            
[korean_petitions] download petitions_2017-10: 12.0MB [00:00, 34.7MB/s]                            
[korean_petitions] download petitions_2017-11: 28.4MB [00:00, 44.6MB/s]                            
[korean_petitions] download petitions_2017-12: 29.0MB [00:02, 11.5MB/s]                            
[korean_petitions] download petitions_2018-01: 43.9MB [00:03, 12.0MB/s]                            
[korean_petitions] download petitions_2018-02: 33.8MB [00:01, 22.9MB/s]                            
[korean_petitions] download petitions_2018-03: 34.3MB [00:00, 42.1MB/s]                            
[korean_petitions] download petitions_2018-04: 35.5MB [00:00, 60.1MB/s]                            
[korean_petitions] download petitions_2018-05: 37.5MB [00:00, 44.4MB/s]                            


In [4]:
petitions = corpus.get_all_texts()
type(petitions), len(petitions)

(list, 433631)

In [8]:
print(petitions[10010])

안전한 국가. 피해자가 정말로 잊어내고 새출발 할수 있게..도와주는 나라..두려움에 떨지않게 신속하게 대처해주시고 제발..부탁드립니다..


In [23]:
import os

os.makedirs("data/petitions", exist_ok=True)
with open("data/petitions/petitions_corpus.txt", "wt", encoding="utf-8") as fw:
    for p in petitions:
        fw.write(p+"\n")

In [24]:
with open("data/petitions/petitions_corpus.txt", "rt", encoding="utf-8") as fr:
    load_petitions = fr.read()

In [32]:
print(len(load_petitions))
load_petitions[:100]


222421175


"안녕하세요. 현재 사대, 교대 등 교원양성학교들의 예비교사들이 임용절벽에 매우 힘들어 하고 있는 줄로 압니다. 정부 부처에서는 영양사의 영양'교사'화, 폭발적인 영양'교사' 채용,"

In [37]:
type(load_petitions)

str

# 6. Hugging Face tokenizers 패키지 사용해 토큰화 처리

- Hugging face는 
  - Subword방식의 토크나이저를 생성할 수 있는 알고리즘을 제공한다. `tokenizers` library를 이용해 사용할 수 있다.
  - GPT, BERT등 다양한 LLM 모델에서 사용된 Pretrained Tokenizer들을 제공한다. `transformers` Library를 이용해 사용할 수 있다.
- 설치: `pip install tokenizers`
- Documentation: https://huggingface.co/docs/tokenizers/index
- Tokenizer 생성
    - 토큰화 알고리즘을 지정해 instance 생성.
- Trainer 생성
    - 학습 파라미터를 설정해서 instance 생성
- Tokenizer 학습
    - train() 메소드: 학습 text 파일 경로를 지정해서 학습
    - train_from_iterator() 메소드: 학습할 string들을 iterator를 통해 제공.
- https://github.com/huggingface/tokenizers

In [38]:
!uv pip install tokenizers

Checked 1 package in 10ms


In [ ]:

import time

from tokenizers import Tokenizer # 문서를 토큰화(어휘사전을 바탕으로)하는 객체
from tokenizers.models import BPE # 토큰화 서브워드 알고리즘
from tokenizers.pre_tokenizers import Whitespace # 서브워드 방식으로 토큰화하기 전에 문서를 미리 토큰화할 때 사용하는 방식
from tokenizers.trainers import BpeTrainer # 토큰나이저 학습시키는 트레이너 객체(토큰나이저모델 학습기)


In [ ]:
# 토큰나이저 생성
tokenizer = Tokenizer(
    BPE(unk_token="[UNK]"), #unk 토큰동록해서 생성. unk-unknown 토큰: 어휘사전에 없는 토큰 (단어)
)
# 프리토큰나이저를 추가
tokenizer.pre_tokenizer = Whitespace()


In [216]:
# 트레이너 생성
# 트레이너 역할 - 토큰나이저 학습 -> 어휘사전을 생성
trainer = BpeTrainer(
    vocab_size=20000, # 어휘사전
    min_frequency=10, # 사전에 추가할 때 최소 출연 횟수(빈도수)
    special_tokens=["[UNK]", "[PAD]"], # 어휘사전에 추가할 특수목적 토큰들 설정
                                       # unk_token에 설정한 것은 반드시 추가한다.   
    continuing_subword_prefix="##", # 연결 토큰과 어절 시작 토큰을 분리해서 저장. 연결토큰 팡에 붙일 접두어 설정 . 워드피스에는 디폴트로 지정
)
# - 스페셜token(특수목적ㅌㅌtoken)
# - 토큰나이저나 언어처리모델이 특정 목적으로 사용할 토큰을 말한다.
# - 학습을 통해 어휘사전에 추가되는 것이 아니라 사람이 직접 추가한디.
# - 보통 스페셜 토큰은 []나 <>로 묶어준다.
# - 대표적인 스페셜token
# - [UNK] <unk>:OOV token 표시(어휘사전에 없는ㅌㅌtoken)
# - [PAD]: 패딩을 표시하는 token, token 갯수를 맞추기 위해서 개수가 모자랄 때token
# - [SOS]: 문장의 시작을 표시하는 token
# - [EOS]: 문장의 끝을 표시하는 token
# - <cls>: 문서의 시작과 문서의 의미를 표현하는 token(BERT 모델이 사용하는 특수 token)
# - <mask>: 문서의 일부 내용들을 가리는 용도의 특수 토큰






In [217]:
# 트레이너를 이용해서 토큰나이ㅓ를 학습(어희사전 생성) - 학습싴릴 데이터셋(말뭉치 ) 전닭

s = time.time()
tokenizer.train(["data/petitions/petitions_corpus.txt"], 
                trainer=trainer
)

e = time.time()
print(f"걸린시간: {e-s}초")


걸린시간: 41.770039081573486초


In [218]:
# 토큰나이저를 파일로 저장: 어휘사전 + 토큰나이저 관련 설정(제이슨)

import os

os.makedirs("saved_model/tokenizers", exist_ok=True)
save_path = "saved_model/tokenizers/petitions_bpe.json"

tokenizer.save(save_path)

In [219]:
# 저장된 토큰나이저를 로드해서 토큰나이저 객체 생성

from tokenizers import Tokenizer
load_tokenizer = Tokenizer.from_file(save_path)


In [220]:
tokenizer.get_vocab_size()

20000

In [221]:
tokenizer.get_vocab()
# load_tokenizer.get_vocab()

{'민사소송법': 19538,
 '##뜸': 9759,
 '##γ': 14451,
 '女': 1734,
 '丈': 1171,
 '학교폭력': 19867,
 '##굳': 10655,
 '##故': 10625,
 '肴': 3123,
 '뺑': 5597,
 '늪': 4539,
 '비난': 17877,
 '🇰': 7938,
 '##봐도': 18236,
 '갢': 3928,
 '🌫': 7952,
 '힌': 7652,
 '盈': 2829,
 '##髑': 13359,
 '분석': 19609,
 '하루에': 19345,
 '##뮈': 10212,
 '경찰서': 17548,
 '##績': 12088,
 '않게': 16530,
 '대통령의': 17458,
 'ญ': 322,
 '又': 1539,
 '##..': 14963,
 '##켸': 10996,
 '贴': 3464,
 '焉': 2665,
 '、': 870,
 '##at': 17972,
 '##땔': 11833,
 '가치': 17822,
 '미세먼지가': 18808,
 '눅': 4491,
 '##주': 8230,
 '욮': 6282,
 '##ؤ': 12972,
 '縛': 3042,
 '##🍍': 11541,
 '##뢀': 10341,
 '고용노동': 19838,
 '##助': 10094,
 '遵': 3583,
 '☷': 750,
 '됴': 4710,
 '##비로': 17460,
 '밨': 5368,
 '낮추': 19907,
 '†': 473,
 '##晝': 12274,
 '딷': 4795,
 '##해주셨으면': 19276,
 '報': 1680,
 '##원을': 15147,
 '##志': 11266,
 '이유가': 16413,
 '##刊': 13549,
 '##豫': 13912,
 '##빰': 10058,
 '鷗': 3869,
 '2015년': 17579,
 '병': 5447,
 '##공': 8382,
 '层': 1863,
 '##소송': 16684,
 '친일파': 19919,
 '국정': 16270,
 '##국을': 1977

In [222]:
# 테스트 문장
sports_txt = "프리미어리그 역대 개인 최다골 기록을 보유하고 있는 시어러가 손흥민의 골 결정력을 재차 극찬했다."
petition_txt = "이 글을 쓴 이유는 다름아닌 '전안법'시행 반대를 주장하기 위해서입니다. 먼저, '전안법'은 전기용품 및 생활용품을 판매하는 업체에서 KC인증마크를 의무적으로 받는 것입니다."
comment_txt = "멋진 식사를 즐기기에 좋은 장소 - 채식 메뉴가 정말 훌륭했습니다. 당근 케이크는 아마도 내가 먹어본 디저트 중 최고였을 거예요."

In [223]:
# 토큰화: 문서(스트링) -> 토큰들로 나누기
token_output = tokenizer.encode(sports_txt)
token_output

Encoding(num_tokens=34, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [224]:
# 토큰화 결과 확인
tokens = token_output.tokens
tokens

['프',
 '리',
 '미',
 '어',
 '리',
 '그',
 '역',
 '대',
 '##책',
 '최',
 '다',
 '골',
 '##“',
 '을',
 '##샘',
 '##산',
 '##습',
 '시',
 '어',
 '러',
 '가',
 '손',
 '흥',
 '민',
 '의',
 '골',
 '##셈',
 '##c',
 '재',
 '차',
 '극',
 '찬',
 '##뷰',
 '.']

In [225]:
# 토큰 아이디로 조회(정수)
token_ids = token_output.ids
print(token_ids)

[7390, 5123, 5330, 6140, 5123, 4128, 6180, 4589, 8414, 6891, 4563, 4027, 9449, 6364, 9442, 8209, 8219, 5908, 6140, 4980, 3902, 5802, 7638, 5332, 6384, 4027, 8940, 8644, 6449, 6793, 4129, 6795, 8565, 15]


In [234]:
token_output2 = tokenizer.encode(petition_txt)
print(token_output2)
token_output2.tokens

Encoding(num_tokens=52, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


['이',
 '##황',
 '쓴',
 '##>',
 '다',
 '름',
 '##러',
 "'",
 '전',
 '안',
 '법',
 "'",
 '##레',
 '##훈',
 '를',
 '##渠',
 '##삼',
 '##맞',
 '##데',
 '.',
 '##G',
 ',',
 "'",
 '전',
 '안',
 '법',
 "'",
 '은',
 '##뇨',
 '용',
 '품',
 '및',
 '##닐',
 '용',
 '품',
 '을',
 '##옵',
 '##의',
 '##좋',
 '##되',
 'K',
 'C',
 '인',
 '증',
 '마',
 '크',
 '를',
 '##앞',
 '##화',
 '##반',
 '##득',
 '.']

In [227]:
token_output2.ids

[6396,
 8577,
 6044,
 9246,
 4563,
 5107,
 8323,
 8,
 6476,
 6073,
 5412,
 8,
 8656,
 8658,
 5101,
 8953,
 8301,
 8580,
 8212,
 15,
 8855,
 13,
 8,
 6476,
 6073,
 5412,
 8,
 6360,
 9353,
 6278,
 7375,
 5345,
 8437,
 6278,
 7375,
 6364,
 9118,
 8208,
 8692,
 8210,
 44,
 36,
 6400,
 6625,
 5140,
 7091,
 5101,
 8521,
 8253,
 8546,
 8299,
 15]

In [228]:
# 개별 토큰 문자령의 아이디 조회
tokenizer.token_to_id("이유는")

15994

In [230]:
# 개별 아이디의 문자열  조회
tokenizer.id_to_token(15994)

'이유는'

In [231]:
# 토큰아이디 리스트 -> 문자열(문서 스티링)
tokenizer.decode(token_ids)

'프 리 미 어 리 그 역 대 ##책 최 다 골 ##“ 을 ##샘 ##산 ##습 시 어 러 가 손 흥 민 의 골 ##셈 ##c 재 차 극 찬 ##뷰 .'

In [232]:
# 원형복구가 안된다.
" ".join(tokens)

'프 리 미 어 리 그 역 대 ##책 최 다 골 ##“ 을 ##샘 ##산 ##습 시 어 러 가 손 흥 민 의 골 ##셈 ##c 재 차 극 찬 ##뷰 .'

In [233]:
sports_txt

'프리미어리그 역대 개인 최다골 기록을 보유하고 있는 시어러가 손흥민의 골 결정력을 재차 극찬했다.'

In [ ]:

# 워드피스 방식으로 토크나이저 학습
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordPieceTrainer
 
import time



In [178]:
#토큰나이저 생성 - 모델을 넣어서 생성(워드피스)
w_tokenizer = Tokenizer(
    WordPiece(unk_token="<unk>"),

)

# 프리토큰나이저 설정
w_tokenizer.pre_tokenizer = Whitespace()


In [179]:
# 트레이너 설정
w_trainer = WordPieceTrainer(
    vocab_size=20000,
    special_tokens=["<unk>", "<pad>", "<sos>", "<eos>", "<mask>"],
    
)

In [184]:
s = time.time()
w_tokenizer.train(
    ["data/petitions/petitions_corpus.txt"],
    trainer = w_trainer
)
print(f"걸린시간: {time.time() -s} 초")

걸린시간: 41.88347029685974 초


In [190]:
save_path = "saved_model/tokenizers/petitions_wordpiece.json"
w_tokenizer.save(save_path)

In [191]:
load_w_tokenizer = Tokenizer.from_file(save_path)

In [ ]:
w_tokenizer.get_vocab_size()

20000

In [196]:
w_tokenizer.get_vocab()

{'##법이': 16437,
 '담배를': 18633,
 'I': 45,
 '\ue904': 7688,
 '##洞': 12031,
 '##頒': 12718,
 '收': 2238,
 '굳이': 18084,
 '##조치': 17840,
 '##딯': 11056,
 '불법': 15096,
 '曨': 2331,
 '🐜': 7999,
 'ض': 289,
 '씽': 6070,
 '##別': 11241,
 '뇌': 4482,
 '##輪': 11011,
 '務': 1484,
 '##퇫': 13656,
 '껶': 4227,
 '##ج': 10457,
 '##쐐': 10694,
 '##選': 13224,
 '대화': 18036,
 '납부': 16745,
 '囑': 1630,
 '수립': 19609,
 'ｏ': 7913,
 '##弟': 13819,
 '암': 6087,
 '##矣': 11303,
 '##아가': 16448,
 '悔': 2066,
 'z': 94,
 '폅': 7332,
 '##捨': 10196,
 '紋': 3008,
 '툻': 7222,
 '##嘉': 14359,
 '임시': 18986,
 '##쉬': 8906,
 '##▪': 13810,
 '각': 3906,
 '##믑': 13390,
 '候': 1325,
 '##浜': 12768,
 '##究': 9933,
 '되면': 15927,
 '銀': 3642,
 '##異': 9923,
 '##자들을': 16343,
 '##乎': 12450,
 '##시간': 15017,
 '뮴': 5321,
 '##수있': 15729,
 '욍': 6268,
 '퉌': 7227,
 '##그룹': 19700,
 '븍': 5546,
 '몈': 5232,
 '##患': 9481,
 '岐': 1880,
 '레': 5001,
 '##샜': 10293,
 '酉': 3615,
 '##계에': 19399,
 '쁠': 5683,
 '##가지고': 18701,
 '##뤘': 10156,
 '국민에게': 16712,
 '##갰': 9321,
 '##誤': 12

In [197]:


# '##법이': 16437

w_tokenizer.token_to_id('##법이')

16437

In [200]:
w_tokenizer.id_to_token(16437)

'##법이'

In [202]:
# 인코딩(토큰화) 디코딩(문서화)
encoding = w_tokenizer.encode(sports_txt)
encoding

Encoding(num_tokens=34, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [203]:
# 토큰들 조회
encoding.tokens

['프',
 '##리',
 '##미',
 '##어',
 '##리',
 '##그',
 '역',
 '##대',
 '개인',
 '최',
 '##다',
 '##골',
 '기록',
 '##을',
 '보유',
 '##하고',
 '있는',
 '시',
 '##어',
 '##러',
 '##가',
 '손',
 '##흥',
 '##민',
 '##의',
 '골',
 '결정',
 '##력을',
 '재',
 '##차',
 '극',
 '##찬',
 '##했다',
 '.']

In [204]:
encoding.ids

[7393,
 8232,
 8233,
 8323,
 8232,
 8212,
 6183,
 8276,
 15090,
 6894,
 8240,
 8995,
 16824,
 8218,
 16336,
 14895,
 14907,
 5911,
 8323,
 8491,
 8229,
 5805,
 8681,
 8226,
 8228,
 4030,
 15625,
 15271,
 6452,
 8278,
 4132,
 8897,
 15375,
 18]

In [251]:
# 인코딩 아이디 목록으로 다시 문자열로 변환 디코딩
result = w_tokenizer.decode(encoding.ids)
result

'##쨋 ͂ 덪 8 쉰 Ⅲ ➾ ; 型 ##꼈 ##ㄱ < N £ ┌ ( 쨨 & 型 ⅸ 车 ŭ ̀ ㉡ 妙 ย !'

In [ ]:
result2 =""
for token_id in encoding.ids:
    token = w_tokenizer.id_to_token(token_id)
    if token.startswith("##"):
        result2 += token[2:]
    else: # 어절 시작 토큰
        result2 += " "+token
    

print(result2.strip())


프리미어리그 역대 개인 최다골 기록을 보유하고 있는 시어러가 손흥민의 골 결정력을 재차 극찬했다 .


In [215]:
sports_txt

'프리미어리그 역대 개인 최다골 기록을 보유하고 있는 시어러가 손흥민의 골 결정력을 재차 극찬했다.'

In [236]:
# 유니그램 토크나이저 학습
# 유니그램 방식으로 토크나이저 학습
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import UnigramTrainer
 
import time

In [237]:
#토큰나이저 생성 - 모델을 넣어서 생성(유니그램)
u_tokenizer = Tokenizer(
    Unigram(), # 언노운 토큰을 설정하지 않는다. 트레이너의 스페셜 토큰중에 0번 토큰을 언노운토큰으로 사용한다.

)

# 프리토큰나이저 설정
u_tokenizer.pre_tokenizer = Whitespace()

In [ ]:
# 트레이너 설정
u_trainer = UnigramTrainer(
    vocab_size=20000,
    special_tokens=["<unk>", "<pad>", "<sos>", "<eos>", "<mask>"], # 여기서 언노운토큰 지정
    min_frequency = 10,
    # continuing_subword_prefix="##",
)

In [254]:
s = time.time()
u_tokenizer.train(
    ["data/petitions/petitions_corpus.txt"],
    trainer = u_trainer
)
print(f"걸린시간: {time.time() -s} 초")

걸린시간: 250.9587836265564 초


In [255]:
save_path = "saved_model/tokenizers/petitions_unigram.json"
u_tokenizer.save(save_path)

In [256]:
load_u_tokenizer = Tokenizer.from_file(save_path)

In [ ]:
##################################################
# 유니그램의 확률값이 표시된다. (로그값을 취했다.)
##################################################

In [258]:
u_tokenizer.get_vocab_size()

20000

In [269]:
u_tokenizer.get_vocab()

{'21': 2390,
 '중국인들': 10937,
 '잣': 13175,
 '잔업': 11190,
 '비공개': 10413,
 '코스닥': 6891,
 '고통속에': 11617,
 '휴가를': 7530,
 '뚯': 15830,
 '㉥': 16803,
 '―': 13297,
 '约': 17343,
 '10억': 5982,
 '\uf125': 15515,
 '안돼': 4801,
 '±': 13790,
 '버려야': 9804,
 '통보를': 7493,
 'で': 14377,
 '학생들이': 1531,
 '야동': 9066,
 '\uf08d': 18118,
 '혈': 2211,
 '膠': 18552,
 '앤': 9568,
 '직접': 503,
 'ヅ': 16719,
 '박정희': 3665,
 '우리들의': 9952,
 '업무': 641,
 '눅': 13490,
 '공무원들': 2632,
 '운전': 1214,
 '遲': 19004,
 '박근혜': 993,
 '묻': 3978,
 '그렇다고': 2592,
 '랔': 18953,
 '뻉': 14631,
 '斥': 14910,
 '以': 13945,
 '연': 257,
 '하듯': 6822,
 '떽': 15234,
 '떔': 15494,
 '쩻': 16720,
 '고시': 2200,
 '가족들은': 9950,
 '砲': 16184,
 '익숙': 11768,
 '뀸': 17990,
 '모두에게': 7587,
 '앵무새': 12839,
 '눈높이': 11214,
 '2013년': 4667,
 '委': 15172,
 '퇫': 16864,
 '고등학생': 4254,
 '전력': 4064,
 '과실': 4752,
 '싸이트': 11820,
 '유리한': 7349,
 '국회의원님들': 8956,
 '해주지': 4159,
 '살기좋은': 9353,
 '즉각': 3176,
 'y': 1364,
 '살아남': 8150,
 '흐지부지': 12456,
 '준비하는': 8790,
 '몿': 16074,
 '‘': 284,
 '꿑': 17076

In [261]:
# '여성에게': 7517
u_tokenizer.token_to_id("여성에게")

7517

In [262]:
u_tokenizer.id_to_token(7517)

'여성에게'

In [270]:
encoding = u_tokenizer.encode(comment_txt)
encoding

Encoding(num_tokens=34, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [271]:
encoding.tokens

['멋진',
 '식사',
 '를',
 '즐기',
 '기에',
 '좋은',
 '장소',
 '-',
 '채',
 '식',
 '메뉴',
 '가',
 '정말',
 '훌륭',
 '했습니다',
 '.',
 '당',
 '근',
 '케이',
 '크',
 '는',
 '아마도',
 '내가',
 '먹어',
 '본',
 '디',
 '저',
 '트',
 '중',
 '최고',
 '였',
 '을',
 '거예요',
 '.']

In [272]:
encoding.ids

[11930,
 4707,
 14,
 5688,
 1050,
 320,
 2278,
 69,
 510,
 252,
 12447,
 12,
 134,
 12592,
 174,
 5,
 161,
 842,
 6651,
 1090,
 13,
 6337,
 654,
 5653,
 277,
 1178,
 137,
 424,
 59,
 1715,
 2127,
 8,
 10839,
 5]

In [273]:
# 인코딩 아이디 목록으로 다시 문자열로 변환 디코딩
result = u_tokenizer.decode(encoding.ids)
result

'멋진 식사 를 즐기 기에 좋은 장소 - 채 식 메뉴 가 정말 훌륭 했습니다 . 당 근 케이 크 는 아마도 내가 먹어 본 디 저 트 중 최고 였 을 거예요 .'

In [267]:
result2 =""
for token_id in encoding.ids:
    token = u_tokenizer.id_to_token(token_id)
    if token.startswith("##"):
        result2 += token[2:]
    else: # 어절 시작 토큰
        result2 += " "+token
    

print(result2.strip())

프리 미 어리 그 역대 개인 최 다 골 기록을 보유하고 있는 시 어 러 가 손흥민 의 골 결정 력을 재 차 극 찬 했다 .
